# Chapter 4: Agent Deployment and Responsible Development

**Book:** *Agents* by Imran Ahmad (Packt, 2026)  
**Author:** Imran Ahmad  
**Chapter:** 4 — Agent Deployment and Responsible Development  

This notebook provides interactive demonstrations of every major concept
in Chapter 4, covering scaling, cost optimization, resilience, architecture,
security, and ethical fairness. All cells run in **Simulation Mode** by default —
no API key is required.

---

In [ ]:
# =============================================================================
# Cell 0: Setup & Environment Detection
# Author: Imran Ahmad
# Ref: Chapter 4 preamble — environment configuration
# =============================================================================

import sys
import os
import random
import time
import json

# Ensure src/ is importable regardless of working directory
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, os.getcwd())

# --- API Key Resolution: .env → getpass → Simulation Mode ---
OPENAI_API_KEY = None
SIMULATION_MODE = True

# Step 1: Try .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
except ImportError:
    pass

# Step 2: If no key yet, try getpass (only in interactive terminals)
if not OPENAI_API_KEY or OPENAI_API_KEY == "your-api-key-here":
    OPENAI_API_KEY = None
    if hasattr(sys, "ps1") or sys.stdin.isatty():
        try:
            import getpass
            key = getpass.getpass("Enter OpenAI API key (or press Enter for Simulation Mode): ")
            if key.strip():
                OPENAI_API_KEY = key.strip()
        except Exception:
            pass

# Step 3: Determine mode
if OPENAI_API_KEY:
    SIMULATION_MODE = False

# --- Import project modules ---
from src.agent_utils import (
    AgentLogger, fail_gracefully, CostTracker,
    CircuitBreaker, InputValidator, format_table, logger,
)
from src.mock_llm import MockLLM, SyntheticDataFactory, RESPONSE_BANK

# --- Initialize shared components ---
mock_llm = MockLLM(latency_ms=100)
cost_tracker = CostTracker(budget_ceiling=1.00)
agent_logger = AgentLogger()

# --- Mode Banner ---
if SIMULATION_MODE:
    agent_logger.info("=" * 60)
    agent_logger.success("SIMULATION MODE ACTIVE")
    agent_logger.info("All responses use chapter-derived mock data.")
    agent_logger.info("No API key required. Every mock value is traceable")
    agent_logger.info("to a specific page, table, or figure in Chapter 4.")
    agent_logger.info("=" * 60)
else:
    agent_logger.info("=" * 60)
    agent_logger.success("LIVE MODE ACTIVE — OpenAI API key detected.")
    agent_logger.info("=" * 60)

print(f"\nPython {sys.version}")
print(f"Simulation Mode: {SIMULATION_MODE}")


---
## Section 4.1: Agent Typology Simulator

**Ref:** Figure 4.1, pp. 3–6

Different agent types exhibit distinct computational and behavioral patterns,
directly influencing their infrastructure dependencies. This cell maps each
typology — reactive, deliberative, hybrid, and multi-agent — to its deployment
target, execution mode, and coordination mechanism as defined in Figure 4.1.

In [ ]:
# =============================================================================
# Cell 1: Section 4.1 — Agent Typology Simulator
# Author: Imran Ahmad
# Ref: Figure 4.1, pp. 3–6 — Infrastructure alignment with agent typologies
# =============================================================================

agent_logger.section_header("4.1", "Agent Typology Simulator")

agent_types = ["reactive", "deliberative", "hybrid", "multi_agent"]

@fail_gracefully(
    fallback_value={
        "execution_mode": "stateless_functions",
        "deployment_target": "serverless_edge",
        "coordination": "event_trigger_http_sqs_webhook",
    },
    section_ref="4.1",
)
def get_infrastructure_recommendation(agent_type):
    """Retrieve infrastructure profile for a given agent typology.

    Author: Imran Ahmad
    Ref: Section 4.1, Figure 4.1, pp. 3–6
    """
    return mock_llm.get_infrastructure_profile(agent_type)


# --- Display infrastructure profiles for all agent types ---
headers = ["Agent Type", "Execution Mode", "Deploy Target", "Coordination", "Use Cases"]
rows = []

for atype in agent_types:
    profile = get_infrastructure_recommendation(atype)
    use_cases = profile.get("use_cases", [])
    use_case_str = ", ".join(uc.replace("_", " ") for uc in use_cases[:2])
    rows.append([
        atype.replace("_", " ").title(),
        profile.get("execution_mode", "N/A").replace("_", " "),
        profile.get("deployment_target", "N/A").replace("_", " "),
        profile.get("coordination", "N/A").replace("_", " "),
        use_case_str,
    ])

print("\nInfrastructure Alignment with Agent Typologies (Figure 4.1, pp. 3–6):\n")
print(format_table(headers, rows))

# --- Track costs ---
cost_tracker.record("mock-gpt-3.5", 80, 0.004)
agent_logger.success("Section 4.1 complete — all four typologies profiled.")


---
## Section 4.2: Cost-Aware Model Router

**Ref:** Figure 4.2, pp. 7–10

Deploying AI agents presents unique economic challenges due to their stochastic
reasoning and variable inference complexity. This cell demonstrates the five
interconnected cost optimization strategies from Figure 4.2: model selection
and routing, tiered architecture, response caching, budget enforcement, and
monitoring.

In [ ]:
# =============================================================================
# Cell 2: Section 4.2 — Cost-Aware Model Router
# Author: Imran Ahmad
# Ref: Figure 4.2, pp. 7–10 — Interconnected cost optimization strategies
# =============================================================================

agent_logger.section_header("4.2", "Cost-Aware Model Router")

# --- Generate synthetic query dataset ---
queries = SyntheticDataFactory.agent_queries(n=50, seed=42)
agent_logger.info(f"Generated {len(queries)} synthetic queries for routing demo.")

# --- Cost-aware routing functions ---

@fail_gracefully(fallback_value=None, section_ref="4.2")
def check_cache(query_text):
    """Check response cache before incurring inference cost.

    Author: Imran Ahmad
    Ref: Section 4.2, p. 8 — Response Caching & Output Reuse
    """
    return mock_llm.check_cache(query_text)


@fail_gracefully(
    fallback_value={"tier": 1, "model": "mock-gpt-3.5", "cost": 0.002},
    section_ref="4.2",
)
def route_query(query_text, budget_status):
    """Route query through tiered architecture with budget awareness.

    Author: Imran Ahmad
    Ref: Section 4.2, pp. 7–10
    """
    # Budget-exceeded degradation (p. 8)
    if budget_status == "degraded":
        return dict(RESPONSE_BANK["4.2_budget_exceeded"])

    # Cache check first (p. 8)
    cached = mock_llm.check_cache(query_text)
    if cached:
        return cached

    # Classify intent and assess confidence
    intent_result = mock_llm.classify_intent(query_text)
    confidence = mock_llm.assess_confidence(query_text)

    # Route to appropriate tier (p. 10)
    return mock_llm.route_to_tier(intent_result["intent"], confidence)


# --- Process a sample of queries through the router ---
router_cost_tracker = CostTracker(budget_ceiling=0.50)
tier_counts = {1: 0, 2: 0, 3: 0}
cache_hits = 0
total_tokens = 0
sample_size = 15

agent_logger.info(f"Processing {sample_size} queries through tiered router...\n")

for i, q in enumerate(queries[:sample_size]):
    budget_status = router_cost_tracker.check_budget()
    result = route_query(q["query_text"], budget_status)

    if result and result.get("cache_status") == "hit":
        cache_hits += 1
        label = "CACHE"
        tier = 0
    elif result and result.get("degraded"):
        label = "DEGRADED"
        tier = 0
    else:
        tier = result.get("tier", 1) if result else 1
        tier_counts[tier] = tier_counts.get(tier, 0) + 1
        label = f"Tier {tier}"

    tokens = result.get("tokens", 0) if result else 0
    cost = result.get("cost", 0.0) if result else 0.0
    total_tokens += tokens

    model = result.get("model", "unknown") if result else "unknown"
    router_cost_tracker.record(model, tokens, cost)

    # Show first 5 queries in detail
    if i < 5:
        print(f"  [{label:>8}] {q['query_text'][:60]:<60} "
              f"→ {tokens:>4} tokens, ${cost:.4f}")

print(f"  ... ({sample_size - 5} more queries processed)")

# --- Display routing summary ---
print(f"\n--- Routing Summary (Figure 4.2 strategies in action) ---")
print(f"  Tier 1 (lightweight):  {tier_counts.get(1, 0)} queries")
print(f"  Tier 2 (intermediate): {tier_counts.get(2, 0)} queries")
print(f"  Tier 3 (high-accuracy):{tier_counts.get(3, 0)} queries")
print(f"  Cache hits:            {cache_hits}")
print(f"  Total mock tokens:     {total_tokens}")

# --- Cost dashboard ---
print(f"\n{router_cost_tracker.summary()}")

# Record to global tracker
cost_tracker.record("tiered-routing-demo", total_tokens, 0.050)
agent_logger.success("Section 4.2 complete — tiered routing with cost tracking demonstrated.")


---
## Section 4.3: Circuit Breaker and Resilience Patterns

**Ref:** Table 4.1, pp. 11–15

High-throughput agent systems require resilience patterns to handle unique
failure modes. This cell demonstrates the circuit breaker pattern from the
book's code (pp. 14–15) using five mock endpoints with varying reliability,
showing state transitions from closed → open → half_open → closed.

In [ ]:
# =============================================================================
# Cell 3: Section 4.3 — Circuit Breaker & Resilience Patterns
# Author: Imran Ahmad
# Ref: Table 4.1, pp. 11–15 — Resilience patterns for high-throughput systems
# =============================================================================

agent_logger.section_header("4.3", "Circuit Breaker & Resilience Patterns")

# --- Display resilience patterns table (Table 4.1, p. 12) ---
print("Resilience Patterns for High-Throughput Agent Systems (Table 4.1, p. 12):\n")
resilience_headers = ["Pattern", "Purpose", "Example Tools"]
resilience_rows = [
    ["Circuit Breakers", "Block failing services to avoid cascade", "Hystrix, Istio, Sentinel"],
    ["Bulkheads", "Isolate failure domains per agent type", "K8s namespaces, Helm charts"],
    ["Timeout + Retry", "Fail fast and reroute intelligently", "Tenacity (Python), Resilience4j"],
    ["Failover Models", "Fallback to cached or distilled responses", "Prompt chaining, Decision DAGs"],
]
print(format_table(resilience_headers, resilience_rows))

# --- Circuit breaker demo with mock endpoints ---
print("\n\n--- Circuit Breaker Demo (extending book code pp. 14–15) ---\n")

endpoints = SyntheticDataFactory.tool_endpoints(n=5)
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=2.0)

# Set deterministic seed for reproducible demo
random.seed(42)

@fail_gracefully(
    fallback_value={"status": "unavailable", "fallback": True},
    section_ref="4.3",
)
def call_external_tool(endpoint_url):
    """Call an external tool API through the circuit breaker.

    Author: Imran Ahmad
    Ref: Section 4.3, pp. 14–15 — Circuit breaker wrapper
    """
    def _do_call():
        return mock_llm.call_tool(endpoint_url)
    return breaker.call(_do_call)


for ep in endpoints:
    print(f"\n  Endpoint: {ep['url']} (reliability={ep['reliability']:.0%})")
    agent_logger.info(f"Testing {ep['url']}...")

    for attempt in range(4):
        result = call_external_tool(ep["url"])
        status = result.get("status", "unknown") if result else "error"
        is_fallback = result.get("fallback", False) if result else True
        marker = " [FALLBACK]" if is_fallback else ""
        print(f"    Attempt {attempt + 1}: status={status}{marker} | "
              f"breaker={breaker.state}")

    # Reset breaker between endpoints for clean demo
    breaker.reset()

# --- Show the book's original code pattern for reference ---
print("\n\n--- Book Code Pattern (pp. 14–15): tenacity + circuit breaker ---")
print("""
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=8))
    def _call_tool_api(endpoint, payload):
        response = httpx.post(endpoint, json=payload, timeout=5.0)
        response.raise_for_status()
        return response.json()

    def call_tool(endpoint, payload):
        global CIRCUIT_OPEN, FAILURE_COUNT
        if CIRCUIT_OPEN:
            return {"status": "unavailable", "fallback": True}
        try:
            result = _call_tool_api(endpoint, payload)
            FAILURE_COUNT = 0
            return result
        except RetryError:
            FAILURE_COUNT += 1
            if FAILURE_COUNT >= FAILURE_THRESHOLD:
                CIRCUIT_OPEN = True
            return {"status": "unavailable", "fallback": True}
""")

cost_tracker.record("circuit-breaker-demo", 0, 0.0)
agent_logger.success("Section 4.3 complete — circuit breaker state transitions demonstrated.")


---
## Section 4.4: Microservice Pipeline Simulation

**Ref:** Table 4.2, p. 13

Microservices architecture decomposes agent functions into independently
deployable services. This cell simulates a query flowing through the
five-service pipeline from Table 4.2: Planner → Retriever → Memory Store
→ Execution Engine → Response Synthesizer.

In [ ]:
# =============================================================================
# Cell 4: Section 4.4 — Microservice Pipeline Simulation
# Author: Imran Ahmad
# Ref: Table 4.2, p. 13 — Microservice decomposition for agent functionality
# =============================================================================

agent_logger.section_header("4.4", "Microservice Pipeline Simulation")

# --- Display service decomposition (Table 4.2) ---
print("Microservice Decomposition (Table 4.2, p. 13):\n")
svc_headers = ["Service Name", "Responsibility"]
svc_rows = [
    ["Planner", "Translates intent into executable steps"],
    ["Retriever", "Performs semantic or keyword-based search"],
    ["Memory Store", "Manages embeddings, chat logs, and context graphs"],
    ["Execution Engine", "Interfaces with APIs, tools, or external systems"],
    ["Response Synthesizer", "Composes and formats final agent output"],
]
print(format_table(svc_headers, svc_rows))

# --- Run pipeline simulation ---
print("\n\n--- Pipeline Execution Trace ---\n")

test_queries = [
    "How do circuit breakers improve agent resilience?",
    "What cost optimization strategies does the chapter recommend?",
]

for query in test_queries:
    agent_logger.info(f"Processing: '{query}'")

    @fail_gracefully(
        fallback_value={"stages": [], "error": "Pipeline failed gracefully"},
        section_ref="4.4",
    )
    def run_pipeline(q):
        """Execute the full five-service microservice pipeline.

        Author: Imran Ahmad
        Ref: Section 4.4, Table 4.2, p. 13
        """
        return mock_llm.simulate_microservice_pipeline(q)

    result = run_pipeline(query)

    if result and "stages" in result:
        for stage in result["stages"]:
            svc = stage["service"]
            status = stage["status"]
            agent_logger.info(f"  [{svc}] → {status}")

            # Show key output for each stage
            output = stage.get("output")
            if svc == "Planner" and isinstance(output, list):
                for step in output:
                    print(f"      Step {step['step']}: {step['action']} → {step['service']}")
            elif svc == "Retriever" and isinstance(output, list):
                for chunk in output:
                    print(f"      Score {chunk['score']}: {chunk['text'][:60]}...")
            elif svc == "Response Synthesizer" and isinstance(output, dict):
                final = output.get("final_response", "")
                print(f"      Final: {final[:80]}...")

        agent_logger.success(f"Pipeline completed: {len(result['stages'])} services executed.")
    print()

# Record cost
cost_tracker.record("pipeline-demo", 300, 0.040)
agent_logger.success("Section 4.4 complete — five-service pipeline demonstrated.")


---
## Section 4.5: Threat Detection and Zero Trust

**Ref:** Tables 4.3a/b, pp. 18–21

AI agents introduce a multifaceted attack surface spanning linguistic,
cognitive, and execution layers. This cell processes 20 inputs (10 benign,
10 adversarial) through the threat detection pipeline, classifying each
against the threat taxonomy from Tables 4.3a/b.

In [ ]:
# =============================================================================
# Cell 5: Section 4.5 — Threat Detection & Zero Trust
# Author: Imran Ahmad
# Ref: Tables 4.3a/b, pp. 18–21 — Threat model for agentic AI systems
# =============================================================================

agent_logger.section_header("4.5", "Threat Detection & Zero Trust")

# --- Display threat taxonomy (Table 4.3a, pp. 18–19) ---
print("Threat Taxonomy by Attack-Surface Layer (Table 4.3a, pp. 18–19):\n")
threat_headers = ["Layer", "Threat Types", "Primary Mitigation"]
threat_rows = [
    ["Input-level", "Prompt Injection, Adversarial Inputs", "Input validation, token sanitization"],
    ["Execution-level", "Tool Misuse, Tool Hijacking, Spoofing", "Tool gating, least-privilege access"],
    ["Memory-level", "Memory Leakage, Context Poisoning", "Memory governance, PII filtering"],
]
print(format_table(threat_headers, threat_rows))

# --- Generate and process threat inputs ---
print("\n\n--- Threat Detection Results (20 inputs) ---\n")

threat_inputs = SyntheticDataFactory.threat_inputs(n=20, seed=42)
validator = InputValidator()

correct = 0
total = len(threat_inputs)

@fail_gracefully(
    fallback_value={"threat": "unknown", "risk_level": "high"},
    section_ref="4.5",
)
def classify_threat(text):
    """Classify input through both validator and MockLLM detector.

    Author: Imran Ahmad
    Ref: Section 4.5, Tables 4.3a/b, pp. 18–19
    """
    validation = validator.sanitize(text)
    detection = mock_llm.detect_threat(text)

    # Merge results: take the more severe classification
    if validation["threats_found"]:
        return {
            "threat": validation["threats_found"][0],
            "risk_level": validation["risk_level"],
            "passed_validation": False,
            "source": "validator",
        }
    if detection.get("threat"):
        return {
            "threat": detection["threat"],
            "risk_level": detection["risk_level"],
            "passed_validation": False,
            "source": "detector",
        }
    return {
        "threat": None,
        "risk_level": "low",
        "passed_validation": True,
        "source": "clean",
    }


for inp in threat_inputs:
    result = classify_threat(inp["input_text"])
    detected_threat = result.get("threat") if result else "unknown"
    risk = result.get("risk_level", "unknown") if result else "unknown"
    is_malicious = inp["is_malicious"]

    # Determine correctness
    if is_malicious and detected_threat is not None:
        correct += 1
        status = "TRUE POSITIVE"
    elif not is_malicious and detected_threat is None:
        correct += 1
        status = "TRUE NEGATIVE"
    elif is_malicious and detected_threat is None:
        status = "FALSE NEGATIVE"
    else:
        status = "FALSE POSITIVE"

    # Color-coded output
    text_preview = inp["input_text"][:55]
    if is_malicious and detected_threat:
        agent_logger.error(f"[{status:>14}] {text_preview:<55} → {detected_threat} ({risk})")
    elif not is_malicious and not detected_threat:
        agent_logger.success(f"[{status:>14}] {text_preview:<55} → clean ({risk})")
    else:
        agent_logger.info(f"[{status:>14}] {text_preview:<55} → {detected_threat} ({risk})")

accuracy = correct / total * 100
print(f"\nDetection Accuracy: {correct}/{total} ({accuracy:.0f}%)")

# --- Zero Trust principles (Table 4.4, p. 20) ---
print("\n\nZero Trust Adaptation for Agent Systems (Table 4.4, p. 20):\n")
zt_headers = ["Principle", "Agent-Centric Implementation"]
zt_rows = [
    ["Least Privilege", "Limit access to tools, data, APIs per task"],
    ["Continuous Verification", "Authenticate actions per invocation, not per session"],
    ["Micro-Segmentation", "Isolate capabilities by namespace or identity"],
    ["Behavior Monitoring", "Telemetry + anomaly detection for outlier decisions"],
    ["Immutable Infrastructure", "Deploy agents in hardened, read-only containers"],
]
print(format_table(zt_headers, zt_rows))

cost_tracker.record("threat-detection-demo", 100, 0.005)
agent_logger.success(f"Section 4.5 complete — {total} inputs classified, {accuracy:.0f}% accuracy.")


---
## Section 4.6: Fairness and Bias Audit

**Ref:** Figure 4.3, p. 24

Bias in AI systems reflects societal inequities encoded in training data.
This cell demonstrates the two distinct fairness problems identified on p. 24:
algorithmic fairness (model prediction bias) and deployment-context fairness.
A synthetic loan-application dataset with deliberate bias is audited, mitigated
via post-processing threshold adjustment, and re-evaluated.

In [ ]:
# =============================================================================
# Cell 6: Section 4.6 — Fairness & Bias Audit
# Author: Imran Ahmad
# Ref: Section 4.6, p. 24 — Fairness and bias mitigation
# =============================================================================

agent_logger.section_header("4.6", "Fairness & Bias Audit")

# --- Responsible AI pillars (Figure 4.3, p. 23) ---
print("Responsible AI Systems — Four Pillars (Figure 4.3, p. 23):\n")
pillars_headers = ["Pillar", "Description"]
pillars_rows = [
    ["Transparency", "Enable understanding of agent decisions for trust and accountability"],
    ["Fairness", "Verify equitable treatment across individuals and groups"],
    ["Accountability", "Governance layer: human oversight + automated monitoring"],
    ["Compliance", "External legal framework integrating all other dimensions"],
]
print(format_table(pillars_headers, pillars_rows))

# --- Generate biased loan dataset ---
print("\n\n--- Fairness Audit: Loan Application Dataset ---\n")

loans = SyntheticDataFactory.loan_applications(n=200, seed=42)
agent_logger.info(f"Generated {len(loans)} loan applications (seed=42).")

predictions = [r["approved"] for r in loans]
groups = [r["demographic_group"] for r in loans]

# --- Pre-mitigation evaluation ---
@fail_gracefully(
    fallback_value=dict(RESPONSE_BANK["4.6_fairness_before"]),
    section_ref="4.6",
)
def compute_fairness_metrics(preds, grps):
    """Compute demographic parity and equalized opportunity metrics.

    Author: Imran Ahmad
    Ref: Section 4.6, p. 24
    """
    return mock_llm.evaluate_fairness(preds, grps)


print("PRE-MITIGATION (raw model predictions):\n")
pre_metrics = compute_fairness_metrics(predictions, groups)

if pre_metrics:
    rates = pre_metrics.get("group_approval_rates", {})
    for grp, rate in sorted(rates.items()):
        count = sum(1 for g in groups if g == grp)
        print(f"  Group {grp}: {rate:.1%} approval rate ({count} applicants)")

    dp = pre_metrics.get("demographic_parity_ratio", 0)
    eq_opp = pre_metrics.get("equalized_opportunity_diff", 0)
    verdict = pre_metrics.get("verdict", "")

    print(f"\n  Demographic Parity Ratio:      {dp:.3f}  (threshold: >= 0.80)")
    print(f"  Equalized Opportunity Diff:    {eq_opp:.3f}  (threshold: <= 0.10)")

    if "FAIL" in verdict:
        agent_logger.error(f"Verdict: {verdict}")
    else:
        agent_logger.success(f"Verdict: {verdict}")

# --- Apply mitigation: post-processing threshold adjustment ---
print("\n\nApplying mitigation: post-processing threshold adjustment (p. 24)...\n")

mitigated_predictions = list(predictions)
rng = random.Random(42)

# Boost Group B approvals to reduce disparity
group_b_indices = [i for i, g in enumerate(groups) if g == "B" and predictions[i] == 0]
boost_count = int(len(group_b_indices) * 0.45)  # Adjust ~35% of denials
boost_targets = rng.sample(group_b_indices, min(boost_count, len(group_b_indices)))
for idx in boost_targets:
    mitigated_predictions[idx] = 1

agent_logger.info(f"Adjusted {len(boost_targets)} Group B decisions via threshold recalibration.")

# --- Post-mitigation evaluation ---
print("\nPOST-MITIGATION (after threshold adjustment):\n")
post_metrics = compute_fairness_metrics(mitigated_predictions, groups)

if post_metrics:
    rates = post_metrics.get("group_approval_rates", {})
    for grp, rate in sorted(rates.items()):
        count = sum(1 for g in groups if g == grp)
        print(f"  Group {grp}: {rate:.1%} approval rate ({count} applicants)")

    dp = post_metrics.get("demographic_parity_ratio", 0)
    eq_opp = post_metrics.get("equalized_opportunity_diff", 0)
    verdict = post_metrics.get("verdict", "")

    print(f"\n  Demographic Parity Ratio:      {dp:.3f}  (threshold: >= 0.80)")
    print(f"  Equalized Opportunity Diff:    {eq_opp:.3f}  (threshold: <= 0.10)")

    if "PASS" in verdict:
        agent_logger.success(f"Verdict: {verdict}")
    else:
        agent_logger.error(f"Verdict: {verdict}")

# --- Note on fairness dimensions ---
print("\n--- Key Distinction (p. 24) ---")
print("The chapter identifies two distinct fairness problems:")
print("  1. Algorithmic fairness: bias in model predictions relative to protected attributes")
print("  2. Deployment-context fairness: bias from how the system is operationalized")
print("This demo addresses algorithmic fairness; deployment-context fairness")
print("requires examining access patterns, feedback loops, and SLA enforcement.")

cost_tracker.record("fairness-demo", 50, 0.005)
agent_logger.success("Section 4.6 complete — pre/post-mitigation fairness audit demonstrated.")


---
## Toolchain Reference Explorer

**Ref:** Table 4.5, pp. 27–29

The following table consolidates every major tool cited in Chapter 4.

In [ ]:
# =============================================================================
# Cell 7: Toolchain Reference Explorer
# Author: Imran Ahmad
# Ref: Table 4.5, pp. 27–29 — Toolchain quick reference for agent deployment
# =============================================================================

agent_logger.section_header("Ref", "Toolchain Reference Explorer")

print("Toolchain Quick Reference for Agent Deployment (Table 4.5, pp. 27–29):\n")

tool_headers = ["Tool", "Role in Agent Deployment", "Install / Access"]
tool_rows = [
    ["Docker", "Containerize agent microservices", "docs.docker.com/get-docker"],
    ["Kubernetes", "Orchestrate Pods; autoscaling, health checks", "minikube start (local)"],
    ["Temporal", "Durable workflow orchestration", "pip install temporalio"],
    ["Prefect", "Python-native workflow scheduling", "pip install prefect"],
    ["OpenTelemetry", "Distributed tracing, metrics, logging", "pip install opentelemetry-sdk"],
    ["Apache Kafka", "High-throughput event streaming", "Docker Compose (local)"],
    ["ArgoCD", "GitOps continuous delivery", "helm install argocd"],
    ["Pinecone", "Managed vector DB for long-context memory", "pip install pinecone-client"],
    ["Grafana", "Observability dashboards", "docker run grafana/grafana"],
    ["tenacity", "Retry and circuit breaker logic (Python)", "pip install tenacity"],
    ["Istio", "Service mesh: circuit breakers, mTLS, A/B", "istioctl install"],
]
print(format_table(tool_headers, tool_rows))

agent_logger.success("Toolchain reference displayed — see pp. 27–29 for full documentation links.")


---
## Summary and Cost Dashboard

Aggregated cost tracking and section completion status across all
demonstrations in this notebook.

In [ ]:
# =============================================================================
# Cell 8: Summary & Cost Dashboard
# Author: Imran Ahmad
# Ref: All sections — aggregated monitoring (Section 4.2 principles applied)
# =============================================================================

agent_logger.section_header("Summary", "Cost Dashboard & Completion Status")

# --- Global cost dashboard ---
print(cost_tracker.summary())

# --- Section completion status ---
print("\n\nSection Completion Status:\n")
status_headers = ["Section", "Topic", "Status"]
status_rows = [
    ["4.1", "Agent Typology Simulator", "COMPLETE"],
    ["4.2", "Cost-Aware Model Router", "COMPLETE"],
    ["4.3", "Circuit Breaker & Resilience", "COMPLETE"],
    ["4.4", "Microservice Pipeline Simulation", "COMPLETE"],
    ["4.5", "Threat Detection & Zero Trust", "COMPLETE"],
    ["4.6", "Fairness & Bias Audit", "COMPLETE"],
    ["Ref", "Toolchain Reference Explorer", "COMPLETE"],
]
print(format_table(status_headers, status_rows))

# --- MockLLM usage stats ---
print("\n\nMockLLM Session Statistics:\n")
stats = mock_llm.get_usage_stats()
for key, val in stats.items():
    label = key.replace("_", " ").title()
    print(f"  {label}: {val}")

# --- Closing ---
print("\n")
agent_logger.success("=" * 60)
agent_logger.success("All sections complete. Notebook validated in Simulation Mode.")
agent_logger.success("=" * 60)
print("\nBook: Agents by Imran Ahmad (Packt, 2026)")
print("Chapter 4: Agent Deployment and Responsible Development")
